In [1]:
import os
import glob
import pandas as pd
import rasterio
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# grid + fire + soil

In [3]:
all_points = pd.read_csv("/content/drive/MyDrive/collab/f2/soil_merged_cleaned_without_duplicate.csv")

In [4]:
climat = pd.read_csv("/content/drive/MyDrive/collab/f2/climat.csv")

In [5]:
print("Nombre de points de sol :", len(all_points))
print("Nombre de points de climat :", len(climat))

Nombre de points de sol : 89198
Nombre de points de climat : 89198


In [11]:
climat.isna().sum()

,0
longitude,0
latitude,0
prec_hiver,6
prec_printemps,6
prec_ete,6
prec_automne,6
tmin_hiver,6
tmin_printemps,6
tmin_ete,6
tmin_automne,6


In [12]:
climat[climat.isna().any(axis=1)]

,longitude,latitude,prec_hiver,prec_printemps,prec_ete,prec_automne,tmin_hiver,tmin_printemps,tmin_ete,tmin_automne,tmax_hiver,tmax_printemps,tmax_ete,tmax_automne
1,-8.952,36.994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26659,-9.502,38.782,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41954,-9.167,39.506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43537,-9.087,39.606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43673,-9.087,39.615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51009,-8.917,40.003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
from sklearn.neighbors import BallTree

col = ['prec_hiver', 'prec_printemps', 'prec_ete',
       'prec_automne', 'tmin_hiver', 'tmin_printemps', 'tmin_ete',
       'tmin_automne', 'tmax_hiver', 'tmax_printemps', 'tmax_ete',
       'tmax_automne']
valid = ~climat[col].isna().any(axis=1)
nan = ~valid
# coordonnees en radians longitude	latitude
valid_coords = np.radians(climat.loc[valid, ["longitude", "latitude"]].values)
nan_coords = np.radians(climat.loc[nan, ["longitude", "latitude"]].values)

# arbre
tree = BallTree(valid_coords, metric="haversine")

# indice du plus proche voisin
dist, idx = tree.query(nan_coords, k=1)

# remplacer par la valeur la plus proche
climat.loc[nan,col] = climat.loc[valid,col].values[idx[:, 0]]

In [14]:
climat[climat.isna().any(axis=1)]

,longitude,latitude,prec_hiver,prec_printemps,prec_ete,prec_automne,tmin_hiver,tmin_printemps,tmin_ete,tmin_automne,tmax_hiver,tmax_printemps,tmax_ete,tmax_automne


In [15]:

# --- paramètres ---
seuil_km = 1.0
seuil_rad = seuil_km / 6371.0  # rayon Terre en km

# --- coords en radians (lat, lon) ---
all_rad = np.radians(all_points[["latitude", "longitude"]].values)
climat_rad = np.radians(climat[["latitude", "longitude"]].values)

# --- arbre sur climat ---
tree = BallTree(climat_rad, metric="haversine")

# --- nearest neighbor pour chaque point ---
dist, idx = tree.query(all_rad, k=1)

# distance en km
dist_km = dist[:, 0] * 6371.0

# --- merge par index du plus proche ---
climat_cols = climat.columns.difference(["longitude", "latitude"])
merged = all_points.copy()
merged[climat_cols] = climat.iloc[idx[:, 0]][climat_cols].values

# --- filtrer par seuil ---
merged["dist_climat_km"] = dist_km
merged["match_climat"] = (dist[:, 0] <= seuil_rad).astype(int)

# --- stats ---
print("Total all_points:", len(all_points))
print("Match <= seuil:", merged["match_climat"].sum())
print("No match:", (merged["match_climat"] == 0).sum())
print("Distance min/max (km):", merged["dist_climat_km"].min(), merged["dist_climat_km"].max())

# optionnel: mettre NaN si pas de match
merged.loc[merged["match_climat"] == 0, climat_cols] = np.nan

merged.drop(columns=["dist_climat_km", "match_climat"], inplace=True)

Total all_points: 89198
Match <= seuil: 89198
No match: 0
Distance min/max (km): 0.0003512594419692697 0.07052210857647025


In [16]:
merged.to_csv("/content/drive/MyDrive/collab/f2/soil_climat_fire_grid_merged.csv", index=False)

In [17]:
Elevation = pd.read_csv("/content/drive/MyDrive/collab/f2/elevation.csv")

In [19]:
Elevation.shape

(89198, 3)